# Calcolo Punteggi Amenity per Immobili

Questo notebook calcola i punteggi amenity per gli immobili utilizzando dati POI spaziali.

## Import Required Libraries

Importa le librerie necessarie come json, os, numpy, pandas, sklearn.neighbors e sys.

In [ ]:
import json
import os
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from scipy.stats import percentileofscore
import sys

## Define Constants

Definisce le costanti come EARTH_RADIUS_KM e MAX_DISTANCE_M.

In [ ]:
# Costanti
EARTH_RADIUS_KM = 6371.0
MAX_DISTANCE_M = 2000  # Distanza massima per considerare POI

In [ ]:
# Definizione delle classi per le amenity
amenity_to_class = {}

# Sanità
essenziali_sanita = ["hospital", "doctor", "pharmacy", "clinic", "defibrillator", "blood_donation", "medical_supply", "dialysis"]
accessori_sanita = ["dentist", "physiotherapist", "physiotherapist-osteopathy", "rehabilitation", "laboratory", "optometrist", "audiologist", "hospice", "veterinary"]
superflui_sanita = ["alternative", "psychotherapist"]

for a in essenziali_sanita:
    amenity_to_class[a] = 'essenziali'
for a in accessori_sanita:
    amenity_to_class[a] = 'accessori'
for a in superflui_sanita:
    amenity_to_class[a] = 'superflui'

# Mobilità
essenziali_mobilita = ["bus_stop", "tram_stop", "subway_entrance", "station", "parking", "bicycle_parking"]
accessori_mobilita = ["taxi", "charging_station", "car_sharing", "bicycle_rental"]
superflui_mobilita = ["car_rental"]

for a in essenziali_mobilita:
    amenity_to_class[a] = 'essenziali'
for a in accessori_mobilita:
    amenity_to_class[a] = 'accessori'
for a in superflui_mobilita:
    amenity_to_class[a] = 'superflui'

# Verde
essenziali_verde = ["park", "playground", "recreation_ground"]
accessori_verde = ["garden", "forest", "wood", "nature_reserve"]
superflui_verde = ["scrub", "grass", "allotments"]

for a in essenziali_verde:
    amenity_to_class[a] = 'essenziali'
for a in accessori_verde:
    amenity_to_class[a] = 'accessori'
for a in superflui_verde:
    amenity_to_class[a] = 'superflui'

# Sport
essenziali_sport = ["sports_centre", "pitch"]
accessori_sport = ["swimming_pool", "stadium", "fitness_centre"]
superflui_sport = ["golf_course", "bowling_alley", "water_park", "ice_rink"]

for a in essenziali_sport:
    amenity_to_class[a] = 'essenziali'
for a in accessori_sport:
    amenity_to_class[a] = 'accessori'
for a in superflui_sport:
    amenity_to_class[a] = 'superflui'

# Commerciale
essenziali_commerciale = ["supermarket", "grocery", "convenience", "bakery", "butcher", "greengrocer", "chemist", "funeral_directors", "food", "rice", "pasta", "dairy", "cheese", "seafood", "deli", "frozen_food"]
accessori_commerciale = ["department_store", "mall", "retail", "general", "clothes", "shoes", "shoe_repair", "laundry", "dry_cleaning", "hardware", "doityourself", "trade", "glaziery", "locksmith", "car_repair", "motorcycle_repair", "bicycle", "computer", "mobile_phone", "telecommunication", "optician", "hairdresser", "stationery", "books", "newsagent", "newspaper", "copyshop", "printing", "estate_agent", "travel_agency", "money_lender", "pet", "garden_centre", "florist", "electrical", "appliance", "hvac", "kitchen", "bathroom_furnishing", "furniture", "lighting", "bed", "flooring", "tiles", "paint", "fabric", "curtain", "window_blind", "household_linen", "houseware", "gas", "baby_goods", "vending_machine", "hearing_aids"]
superflui_commerciale = ["alcohol", "wine", "tobacco", "e-cigarette", "cannabis", "bookmaker", "tattoo", "erotic", "massage", "spa", "beauty", "perfumery", "cosmetics", "jewelry", "watches", "gold_buyer", "pawnbroker", "antiques", "art", "auction_house", "craft", "frame", "photo", "camera", "video", "hifi", "musical_instrument", "music", "video_games", "anime", "toys", "model", "games", "hobby", "collector", "numismatics", "hunting", "weapons", "army", "military_surplus", "outpost", "boat", "fishing", "scuba_diving", "water_sports", "sports", "party", "gift", "confectionery", "chocolate", "ice_cream", "pastry", "coffee", "coffee_roasting", "tea", "spices", "nutrition_supplements", "wigs", "second_hand", "charity", "rental", "tool_hire", "car_parts", "tyres", "car", "motorcycle", "scooter", "caravan", "bag", "fashion_accessories", "tailor", "sewing", "pottery", "plaques", "religion", "wholesale", "security", "vacant", "printer_ink", "cartridges", "hairdresser_supply", "brewing_supplies", "radiotechnics", "interior_decoration", "variety_store"]

for a in essenziali_commerciale:
    amenity_to_class[a] = 'essenziali'
for a in accessori_commerciale:
    amenity_to_class[a] = 'accessori'
for a in superflui_commerciale:
    amenity_to_class[a] = 'superflui'

# Educazione
essenziali_educazione = ["school", "kindergarten", "university", "college"]
accessori_educazione = ["library", "research_institute", "community_centre", "museum"]
superflui_educazione = ["driving_school", "music_school"]

for a in essenziali_educazione:
    amenity_to_class[a] = 'essenziali'
for a in accessori_educazione:
    amenity_to_class[a] = 'accessori'
for a in superflui_educazione:
    amenity_to_class[a] = 'superflui'

## Load Immobili Data

Carica il dataset degli immobili da un file Parquet utilizzando pandas.

In [ ]:
def load_immobili_data():
    """Carica il dataset degli immobili."""
    # Aggiungi la root del progetto al path
    sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '../../')))
    from app.core.config import DATASET_FULL
    immobili_path = os.path.join(os.path.dirname('__file__'), '../../', DATASET_FULL)
    immobili_path = os.path.abspath(immobili_path)
    print(f"Caricando immobili da {immobili_path}")
    df = pd.read_parquet(immobili_path)
    print(f"Caricato {len(df)} immobili")
    return df

## Load POIs Data

Carica i dati POI categorizzati da un file JSON.

In [ ]:
def load_pois_data():
    """Carica i POI categorizzati."""
    pois_path = os.path.join(os.path.dirname('__file__'), '../01_pois/pois_by_category.json')
    print(f"Caricando POI da {pois_path}")
    with open(pois_path, 'r', encoding='utf-8') as f:
        pois_by_category = json.load(f)
    print(f"Caricato POI per {len(pois_by_category)} categorie")
    return pois_by_category

## Build Spatial Index

Costruisce un indice spaziale utilizzando BallTree per tutti i POI.

In [ ]:
def build_spatial_index(pois_by_category):
    """Costruisce l'indice spaziale per tutti i POI."""
    all_pois = []
    for category, amenities in pois_by_category.items():
        for amenity, pois in amenities.items():
            for poi in pois:
                all_pois.append({
                    'category': category,
                    'amenity': amenity,
                    'lat': poi['lat'],
                    'lon': poi['lon']
                })

    if not all_pois:
        raise ValueError("Nessun POI trovato")

    # Coordinate in radianti per BallTree
    coords = np.array([[np.radians(p['lat']), np.radians(p['lon'])] for p in all_pois])
    tree = BallTree(coords, metric='haversine')

    print(f"Costruito indice spaziale con {len(all_pois)} POI")
    return tree, all_pois

In [ ]:
# Debug: verifica range coordinate POI
immobili_df = load_immobili_data()
pois_by_category = load_pois_data()
tree, all_pois = build_spatial_index(pois_by_category)

# Verifica range POI
poi_lats = [p['lat'] for p in all_pois]
poi_lons = [p['lon'] for p in all_pois]
print(f"POI - Latitudine: min={min(poi_lats):.6f}, max={max(poi_lats):.6f}")
print(f"POI - Longitudine: min={min(poi_lons):.6f}, max={max(poi_lons):.6f}")

# Verifica range immobili
immobili_lats = immobili_df['latitudine'].values
immobili_lons = immobili_df['longitudine'].values
print(f"\nImmobili - Latitudine: min={immobili_lats.min():.6f}, max={immobili_lats.max():.6f}")
print(f"Immobili - Longitudine: min={immobili_lons.min():.6f}, max={immobili_lons.max():.6f}")

# Verifica se ci sono valori anomali
print(f"\nNumero totale immobili: {len(immobili_df)}")
print(f"Numero totale POI: {len(all_pois)}")

## Calculate Amenity Scores

Calcola i punteggi amenity per ogni immobile utilizzando query spaziali e decadimento esponenziale.

In [ ]:
from scipy.stats import percentileofscore

def calculate_amenity_scores(immobili_df, tree, all_pois, output_path, categories):
    """Calcola i punteggi amenity aggregati per ogni immobile (per categoria, facendo la media delle amenity) e i percentili. Assicura che ogni immobile abbia una riga per ogni categoria, con score 0 se non presente."""
    # Definizione degli alpha per le classi
    alpha1 = 2 * np.log(2)  # Essenziali
    alpha2 = np.log(2)      # Accessori
    alpha3 = 0.5 * np.log(2)  # Superflui
    
    # Pre-calcola category e amenity per ogni POI
    poi_categories = np.array([p['category'] for p in all_pois])
    poi_amenities = np.array([p['amenity'] for p in all_pois])
    
    max_dist_rad = MAX_DISTANCE_M / 1000 / EARTH_RADIUS_KM
    
    results = []
    min_dist = float('inf')
    max_dist_val = -float('inf')
    
    for idx in range(len(immobili_df)):
        immobile = immobili_df.iloc[idx]
        immobile_id = immobile['id']
        immobile_lat = immobile['latitudine']
        immobile_lon = immobile['longitudine']
        
        # Query POI entro MAX_DISTANCE_M con distanze
        query_point = np.array([[np.radians(immobile_lat), np.radians(immobile_lon)]])
        # NOTA: query_radius con return_distance=True restituisce (indices, distances) NON (distances, indices)!
        indices_list, distances_list = tree.query_radius(query_point, r=max_dist_rad, return_distance=True)
        # Coerce a numpy arrays con dtype sicuri
        indices = np.array(indices_list[0], dtype=int)
        distances = np.array(distances_list[0], dtype=float)
        distances_km = distances * EARTH_RADIUS_KM
        
        if indices.size == 0:
            # Se non ci sono POI, aggiungi righe con score 0 per tutte le categorie
            for category in categories:
                results.append({
                    'immobile_id': immobile_id,
                    'category': category,
                    'score': 0.0
                })
            continue
        
        # Aggiorna min e max distanze
        min_dist = min(min_dist, distances.min())
        max_dist_val = max(max_dist_val, distances.max())
        
        # Aggrega punteggi per categoria e amenity
        amenity_scores = {}  # chiave: (category, amenity), valore: somma punteggi
        
        for i, poi_idx in enumerate(indices):
            poi = all_pois[poi_idx]
            classe = amenity_to_class.get(poi['amenity'], 'superflui')
            if classe == 'essenziali':
                alpha = alpha1
            elif classe == 'accessori':
                alpha = alpha2
            else:
                alpha = alpha3
            score_contrib = np.exp(-alpha * distances_km[i])
            
            key = (poi['category'], poi['amenity'])
            if key not in amenity_scores:
                amenity_scores[key] = 0.0
            amenity_scores[key] += score_contrib
        
        # Aggiungi risultati aggregati per questo immobile (media per categoria), assicurando tutte le categorie
        category_scores = {}
        for (category, amenity), total_score in amenity_scores.items():
            if category not in category_scores:
                category_scores[category] = []
            category_scores[category].append(total_score)
        
        for category in categories:
            if category in category_scores:
                avg_score = sum(category_scores[category]) / len(category_scores[category])
            else:
                avg_score = 0.0
            results.append({
                'immobile_id': immobile_id,
                'category': category,
                'score': avg_score
            })
        
        if (idx + 1) % 100 == 0:
            print(f"Elaborati {idx + 1}/{len(immobili_df)} immobili")
    
    # Calcola percentili per categoria
    category_scores_all = {}
    for result in results:
        cat = result['category']
        if cat not in category_scores_all:
            category_scores_all[cat] = []
        category_scores_all[cat].append(result['score'])
    
    for result in results:
        cat = result['category']
        scores_cat = np.array(category_scores_all[cat])
        percentile = percentileofscore(scores_cat, result['score'], kind='rank')
        result['percentile'] = percentile
    
    # Salva tutto in CSV
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_path, index=False)
    
    return results_df, min_dist, max_dist_val

## Save Results

Salva i risultati in un file CSV ed esegue il processo principale.

In [ ]:
def main(subset_size=None):
    print("Inizio calcolo immobili_amenity_scores_aggregated.csv")

    # Carica dati
    immobili_df = load_immobili_data()
    
    # Usa subset se richiesto
    if subset_size:
        immobili_df = immobili_df.head(subset_size)
        print(f"DEBUG: Usando solo {len(immobili_df)} immobili per test")
        output_path = 'immobili_amenity_scores_aggregated_debug.csv'
    else:
        output_path = 'immobili_amenity_scores_aggregated.csv'
    
    pois_by_category = load_pois_data()
    categories = list(pois_by_category.keys())

    # Costruisci indice spaziale
    tree, all_pois = build_spatial_index(pois_by_category)

    # Calcola punteggi con salvataggio parziale
    results_df, min_dist, max_dist = calculate_amenity_scores(immobili_df, tree, all_pois, output_path, categories)

    print(f"\nStatistiche distanze:")
    print(f"  Min distanza: {min_dist * EARTH_RADIUS_KM * 1000:.2f} m")
    print(f"  Max distanza: {max_dist * EARTH_RADIUS_KM * 1000:.2f} m")

    print(f"\nSalvato {output_path} (struttura: immobile_id,category,score,percentile)")
    print("Completato!")

# Esegui il main
main()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 2, 100)
alpha1 = 2 * np.log(2)
alpha2 = np.log(2)
alpha3 = 0.5 * np.log(2)

y1 = np.exp(-alpha1 * x)
y2 = np.exp(-alpha2 * x)
y3 = np.exp(-alpha3 * x)

plt.figure()
plt.plot(x, y1, label=f'alpha = {alpha1:.3f}')
plt.plot(x, y2, label=f'alpha = {alpha2:.3f}')
plt.plot(x, y3, label=f'alpha = {alpha3:.3f}')
plt.xlim(0, 2)
plt.ylim(0, 1)
plt.xlabel('Distanza (km)')
plt.ylabel('Score')
plt.legend()
plt.show()